### 5.4 RunnablePassthrough-입력을 그대로 출력하기

In [1]:
# 먼저, prompt(ChatPromptTemplate)와 model(ChatOpenAI)을 준비
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv("../.env")

prompt = ChatPromptTemplate.from_template('''
다음 문맥만을 고려하여 질문에 답하세요.
문맥: """{context}"""
질문: {question}
''')

model = ChatOpenAI(model="gpt-6-luna")

In [2]:
# 다음으로 Tavily를 LangChain의 Retriever로 사용하기 위한 TavilySearchRetriever를 준비
from langchain_community.retrievers import TavilySearchAPIRetriever

retriever = TavilySearchAPIRetriever(k=3)

In [5]:
# 지금까지 준비한 prompt, model, retriever를 사용해 RAG의 Chain을 구현
from langchain_core.runnables import RunnablePassthrough

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)
output = chain.invoke("서울의 현재 날씨는?")
print(output)

c:\Users\limux\Documents\1. Projects\llm-engineer-roadmap\02-rag-agents-langchain-langgraph\.venv\Lib\site-packages\langsmith\_internal\_addressing.py:104: LangSmithBetaWarning: Addressing runs with `langsmith.address` is in beta and is enabled per workspace. A workspace without it rejects the run, so the trace is lost rather than falling back to a project. The behavior may change without notice.
  _warn_once(


서울은 현재 **5°C로 대체로 맑고 쌀쌀합니다.** 바람은 서풍으로 시속 4km 정도입니다.


### assign-RunnableParallel의 출력에 값 추가하기

In [6]:
import pprint

chain = {
    "question": RunnablePassthrough(),
    "context": retriever,
} | RunnablePassthrough.assign(
    answer=prompt | model | StrOutputParser()
)
output = chain.invoke("서울의 현재 날씨는?")
pprint.pprint(output)

{'answer': '서울은 현재 **5°C로 대체로 맑고**, 체감온도도 5°C입니다. 서풍은 시속 4km 정도로 불고 있습니다.',
 'context': [Document(metadata={'title': '서울특별시, 서울시, 대한민국 현재 날씨 | AccuWeather', 'source': 'https://www.accuweather.com/ko/kr/seoul/226081/current-weather/226081', 'score': 0.81875163, 'id': 'bd4e80-00', 'images': []}, page_content='RealFeel® 13°\n\n선선함\n\nRealFeel Shade™ 13°\n\n선선함\n\n흐리고 오후에 한두 차례 짧은 소나기가 내림\n\n최대 자외선 지수1.0 (좋음)\n\nAccuLumen Brightness Index™1 (어둡게)\n\n바람남남동 11km/h\n\n돌풍26km/h\n\n강수 확률68%\n\n뇌우 확률14%\n\n강수2.2mm\n\n강수량2.2mm\n\n총 강수 시간1.5\n\n총 강수 시간1.5\n\n구름량97%\n\n### 오전\n\n### 오후\n\n## 밤\n\n 10. 29.\n\n8°최저\n\nRealFeel® 9°\n\n쌀쌀함\n\n약간 흐림\n\n바람동남동 6km/h\n\n돌풍22km/h\n\n강수 확률25%\n\n뇌우 확률0%\n\n강수0.0mm\n\n구름량48%\n\n### 저녁\n\n### 밤사이\n\n## 해와 달\n\n 10 시간 45 분\n\n일출 AM 6:53\n\n일몰 PM 5:38\n\n 15 시간 50 분\n\n일출 PM 7:44\n\n일몰 AM 11:34\n\n## 온도 기록\n\n10. 29.\n\n높음\n\n낮음\n\n### 시간별 ### 10일 ### 월 [...] 돌아가기 \n\n# 서울특별시, 서울시\n\n5°C  \n\n현재 위치 사용\n\n 5°\n\n결과를 찾을 수 없습니다.\n\n도시나 우편번호, 관심 지역을 검색해 보세요.\n\n도움말 

In [8]:
# 참고로, assign은 RunnablePassthrough의 클래스 메서드 외에도 Runnable의 인스턴스 메서드로도 제공된다.
# 그래서 앞의 예와 같은 처리를 다음과 같이 작성할 수도 있다.
from langchain_core.runnables import RunnableParallel

chain = RunnableParallel({
    "question": RunnablePassthrough(),
    "context": retriever
}).assign(answer=prompt | model | StrOutputParser())

### pick

In [9]:
# Runnable의 pick 메서드를 사용하면 assign과 반대로 dict의 일부만 선택할 수 있다.
# 앞의 예를 기반으로 question은 출력하지 않고 context와 answer만 출력하는 Chain을 구현하는 예는 다음과 같다.
chain = (
    RunnableParallel({
        "question": RunnablePassthrough(),
        "context": retriever
    }).assign(
        answer=prompt | model | StrOutputParser()
    ).pick(["context", "answer"])
)
output = chain.invoke("서울의 날씨는?")
pprint.pprint(output)

{'answer': '문맥상 서울은 **2026년 10월 28일 현재 맑고, 기온은 약 14°C**입니다. 예보는 출처마다 차이가 있지만, '
           '내일(29일)은 흐리고 가끔 비가 올 가능성이 있습니다.',
 'context': [Document(metadata={'title': '날씨 - 서울특별시 - 14일 예보: 기온, 바람 및 레이더 | Ventusky', 'source': 'https://www.ventusky.com/ko/seoul', 'score': 0.67347145, 'id': '82e031-00', 'images': []}, page_content="기온\n 체감 온도\n 강수량\n 날씨 레이더\n Satellite\n 풍속\n 강풍\n 구름량\n 기압\n CAPE\n 적설량\n 습도\n 바다\n 대기 오염\n\n풍향 에니메이션   \n ECMWF, GFS, ICON, GEM\n\nTwo New High-Resolution Weather Models on Ventusky Model\n\n세계 / 대한민국 / 서울특별시 (Seoul)\n\n37°34'북 / 126°58'동 / 고도 44 m / 07:01 2026.10.28, Asia/Seoul (UTC+9)\n\n 현재 날씨\n 예보\n 해와 달\n\n|  |  |\n --- |\n| 맑은 하늘 | 3.8 °C |\n|  |\n| 풍속   14 km/h |\n\n|  |  |\n --- |\n| 습도 | 36 % |\n| 기압 | 1027 hPa |\n| 시계 | 3 km |\n| 구름량 | 0 % |\n\n기상대 정보 Seoul City, 거리: 1 km (06:00 2026.10.28)\n\n레이더 지도 열기\n\n## 다음 24시간 [...] ## 예보\n\n 목   \n   흐리고 약한 비가 내린다   \n   10 °C\n 금   \n   맑은 하늘   \n   17 °C\n 토   \n   맑은 하늘   \n   15 °C\n 일   \n   흐리고 비   \n  

### astream_events

In [10]:
chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

async for event in chain.astream_events("서울의 현재 날씨는?", version="v2"):
    print(event, flush=True)

{'event': 'on_chain_start', 'data': {'input': '서울의 현재 날씨는?'}, 'name': 'RunnableSequence', 'tags': [], 'run_id': '01a18567-2543-752f-82d0-97a7fd4f2d4e', 'metadata': {}, 'parent_ids': []}
{'event': 'on_chain_start', 'data': {}, 'name': 'RunnableParallel<context,question>', 'tags': ['seq:step:1'], 'run_id': '01a18567-2545-72ef-8e23-2882ce228b3f', 'metadata': {}, 'parent_ids': ['01a18567-2543-752f-82d0-97a7fd4f2d4e']}
{'event': 'on_retriever_start', 'data': {'input': {'query': '서울의 현재 날씨는?'}}, 'name': 'TavilySearchAPIRetriever', 'tags': ['map:key:context'], 'run_id': '01a18567-2547-7204-af76-3761f84a493c', 'metadata': {'ls_retriever_name': 'tavilysearchapi'}, 'parent_ids': ['01a18567-2543-752f-82d0-97a7fd4f2d4e', '01a18567-2545-72ef-8e23-2882ce228b3f']}
{'event': 'on_chain_start', 'data': {}, 'name': 'RunnablePassthrough', 'tags': ['map:key:question'], 'run_id': '01a18567-2548-7312-b527-322746faf577', 'metadata': {}, 'parent_ids': ['01a18567-2543-752f-82d0-97a7fd4f2d4e', '01a18567-2545-72e

In [12]:
# astream_events의 이벤트 종류에 따라 처리를 분기하는 코드를 작성하면 Chain의 중간 값을 출력할 수 있다.
async for event in chain.astream_events("서울의 현재 날씨는?", version="v2"):
    event_kind = event["event"]
    if event_kind == "on_retriever_end":
        print("=== 검색 결과 ===")
        documents = event["data"]["output"]
        for document in documents:
            print(document)
    elif event_kind == "on_parser_start":
        print("=== 최종 출력 ===")
    elif event_kind == "on_parser_stream":
        chunk = event["data"]["chunk"]
        print(chunk, end="", flush=True)

=== 검색 결과 ===
page_content='RealFeel® 13°

선선함

RealFeel Shade™ 13°

선선함

흐리고 오후에 한두 차례 짧은 소나기가 내림

최대 자외선 지수1.0 (좋음)

AccuLumen Brightness Index™1 (어둡게)

바람남남동 11km/h

돌풍26km/h

강수 확률68%

뇌우 확률14%

강수2.2mm

강수량2.2mm

총 강수 시간1.5

총 강수 시간1.5

구름량97%

### 오전

### 오후

## 밤

 10. 29.

8°최저

RealFeel® 9°

쌀쌀함

약간 흐림

바람동남동 6km/h

돌풍22km/h

강수 확률25%

뇌우 확률0%

강수0.0mm

구름량48%

### 저녁

### 밤사이

## 해와 달

 10 시간 45 분

일출 AM 6:53

일몰 PM 5:38

 15 시간 50 분

일출 PM 7:44

일몰 AM 11:34

## 온도 기록

10. 29.

높음

낮음

### 시간별 ### 10일 ### 월 [...] 돌아가기 

# 서울특별시, 서울시

5°C  

현재 위치 사용

 5°

결과를 찾을 수 없습니다.

도시나 우편번호, 관심 지역을 검색해 보세요.

도움말 

서울특별시, 서울시 날씨

오늘 WinterCast 지역 {stormName} 추적기 시간별 10일 레이더 MinuteCast® 월간 대기질 건강 및 활동

전 세계

### 레이더 및 지도 ### 동영상 ### 악천후 기상 ### 허리케인

오늘   시간별   10일   레이더   MinuteCast®   월   대기질   건강 및 활동

목요일, 10월 29일

# 현재 기상

AM 1:52

5°C

대체로 맑음

RealFeel® 5°

쌀쌀함

RealFeel®

5°

바람

서 4km/h

돌풍

5km/h

습도

77%

이슬점

1° C

기압

↓ 1025mb

구름량

22%

가시거리

16km

운저

12200m

## 낮

 10. 29.

